# Pure Pursuit Steering Controller

---

## What Is Pure Pursuit?

Pure Pursuit is one of the most widely used geometric path-tracking algorithms in autonomous driving. It was first introduced in 1985 and remains popular due to its simplicity as well as effectiveness.

The idea behind Pure Pursuit is to compute the steering angle that moves the **rear axle** of the vehicle toward a **goal point** on the path that is a fixed **lookahead distance** ahead of the vehicle.

Think of it like this: when you drive a car, you don't stare at the road directly in front of your bumper — you look some distance ahead and steer toward that point. Pure Pursuit formalises this idea.

---

## Geometric Setup

Given:

- $(x, y)$: rear axle position of the vehicle  
- $\psi$: vehicle heading angle (yaw)  
- $L$ : wheelbase of the vehicle  
- $l_d$ : lookahead distance — how far ahead on the path we pick the goal point  
- $(g_x, g_y)$: the goal point on the path at distance $l_d$ from the vehicle  
- $\alpha$: the angle between the vehicle's heading and the line from the rear axle to the goal point  

---

## Deriving the Steering Angle

The angle $\alpha$ is computed as:

$$
\alpha = \arctan2(g_y - y, \; g_x - x) - \psi
$$

Using the bicycle model, the steering angle $\delta$ that drives the vehicle along a circular arc through the goal point is given by:

$$
\delta = \arctan\left(\frac{2 \, L \, \sin(\alpha)}{l_d}\right)
$$

Where:

- $L$ is the wheelbase (distance between front and rear axles).  
- $l_d$ is the lookahead distance; a larger $l_d$ produces smoother but less responsive tracking, while a smaller $l_d$ tracks the path more tightly but can oscillate.  
- $\alpha$ is the angle between the vehicle heading and the direction to the goal point.  
- $\delta$ is the steering angle, which should be clipped to the maximum steering limits of the vehicle.

---

## Key Tuning Parameter — Lookahead Distance

The lookahead distance $l_d$ is the primary tuning knob:

| Lookahead | Behaviour                         |
|-----------|------------------------------------|
| Small     | Tight tracking, may oscillate      |
| Large     | Smooth steering, may cut corners   |

A common strategy is to make $l_d$ proportional to the vehicle speed:

$$
l_d = k_{dd} \cdot v + l_{d,\min}
$$

Where $k_{dd}$ is a gain and $l_{d,\min}$ is a minimum lookahead to prevent instability at low speeds.

---

## How It Differs from Stanley

| Feature              | Pure Pursuit                      | Stanley                            |
|----------------------|-----------------------------------|------------------------------------|
| Reference point      | Rear axle                         | Front axle                         |
| Target point         | Goal at lookahead distance        | Nearest point on path              |
| Error terms          | Angle to goal ($\alpha$)          | Heading error + cross-track error  |
| Tuning parameter     | Lookahead distance ($l_d$)        | Cross-track gain ($k$)             |

---

## Your Task

Implement the Pure Pursuit steering function using the signature below. Fill in the missing steps — each step is clearly commented.

You may also import this function in your Checkpoint 4 animation notebook to compare its behaviour with Stanley control.


In [1]:
import numpy as np

def pure_pursuit_steering(x, y, yaw, v, waypoints, L=2.5, k_dd=0.5, ld_min=2.0, max_steer=np.radians(30)):
    """
    Pure Pursuit steering controller.

    Args:
        x, y       : rear axle position of the car
        yaw        : vehicle heading angle (in radians)
        v          : vehicle speed
        waypoints  : Nx2 array of path waypoints
        L          : wheelbase (distance between front and rear axles)
        k_dd       : lookahead gain (scales with speed)
        ld_min     : minimum lookahead distance
        max_steer  : steering angle limits (in radians)

    Returns:
        steer      : steering angle in radians
        target_idx : index of the goal waypoint
    """

    # Step 1: Compute the lookahead distance based on speed
    ld = k_dd * v + ld_min


    # Step 2: Find the nearest waypoint to the rear axle
    #   Compute distances from (x, y) to every waypoint, pick the closest index.
    min_dist = float('inf')
    nearest_idx = 0
    for i, (wx, wy) in enumerate(waypoints):
        dist = np.hypot(wx - x, wy - y)
        if dist < min_dist:
            min_dist = dist
            nearest_idx = i


    # Step 3: Search forward from the nearest waypoint to find the goal point
    #   Starting from the nearest index, walk forward along the waypoints until
    #   you find the first waypoint whose distance from (x, y) >= ld.
    #   If none is found, use the last waypoint in the array.
    target_idx = nearest_idx
    while target_idx < len(waypoints):
        wx, wy = waypoints[target_idx]
        dist = np.hypot(wx - x, wy - y)
        if dist >= ld:
            break
        target_idx += 1

    # Step 4: Compute alpha — the angle between the vehicle heading and the
    #   direction from the rear axle to the goal point.
    alpha = np.arctan2(waypoints[target_idx][1] - y, waypoints[target_idx][0] - x) - yaw


    # Step 5: Compute the steering angle using the Pure Pursuit formula:
    #   steer = arctan(2 * L * sin(alpha) / ld)
    steer = np.arctan(2 * L * np.sin(alpha) / ld)

    # Step 6: Clip steering to max_steer
    steer = np.clip(steer, -max_steer, max_steer)

    return steer, target_idx
